# Week 1 · Day 1 — Website Summarizer

A first look at calling an LLM from Python. We'll:

1. Set up the OpenAI client and load our API key.
2. Send a simple chat completion ("tell me a joke").
3. Scrape a real web page down to clean text.
4. Ask the model to summarize that page in markdown.

> **Prerequisite:** copy `.env.example` to `.env` and add your `OPENAI_API_KEY`.

---

## 1. Setup & imports

In [21]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from scraper import clean_html, fetch_website_content

## 2. Load API credentials

`load_dotenv` reads the key/value pairs from our local `.env` file into
environment variables. We fail fast if the OpenAI key is missing.

In [22]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")

## 3. Your first chat completion

The Chat Completions API takes a list of `messages`. Each message has a
`role` (`system`, `user`, or `assistant`) and `content`. Here we send a
single user message and print the model's reply.

In [23]:
message = "Hi GPT it's my first time using you, can you tell me a joke?"
messages = [
    {"role": "user", "content": message},
]

In [24]:
# Instantiate the client. It automatically picks up OPENAI_API_KEY from the
# environment, so no key needs to be passed explicitly.
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,  # type: ignore
)

# The reply text lives at choices[0].message.content.
display(Markdown(f"**GPT:** {response.choices[0].message.content}"))

**GPT:** Sure! Here's a quick one:

Why don’t scientists trust atoms? Because they make up everything!

Want another? I can tailor it to your vibe—puns, dad jokes, knock-knock, or tech humor.

## 4. Scrape a website

Before we can summarize a page, we need its content as plain text.
`fetch_website_content` (defined in [scraper.py](scraper.py)) downloads the
page, strips out scripts/styles/images, and returns the title plus visible
text.

In [25]:
my_website_content = fetch_website_content(url="https://azam-sys.netlify.app/")
print(my_website_content)

Title: Azam Mustufa — Software Engineer

Azam Mustufa — Software Engineer
AZAM.DEV
Work
Experience
Connect
mail
System Operational · IBM, Bangalore
Software Engineer
at IBM
Software Engineer at IBM working on SAP SuccessFactors cloud integrations. I build backend systems with Java, Spring Boot, and Node.js, and focus on quality engineering, test automation, and distributed systems.
281+
LeetCode · Top 13%
5
Projects deployed
3×
Hackathon finalist
View Work
arrow_downward
Connect
Projects
Backend Systems
terminal
Live
Database Backup CLI Infrastructure
Production-grade automation CLI for async database backups — reduces manual deployment and migration time by 80%, handles 50GB migrations under 150MB memory with 99.9% success rate.
TypeScript
MongoDB
MySQL
Node.js
Linux
Bash
GitHub
arrow_outward
hub
Live
Project Management Engine
Scalable REST backend with 25+ endpoints and granular RBAC. API responses optimised to under 120ms via Redis caching. BullMQ async pipelines cut cache latency b

## 5. Summarize the page with an LLM

This is a classic **system + user prompt** pattern:

- The **system prompt** sets the model's role and output format.
- The **user prompt** carries the actual data (the scraped page text).

In [26]:
# Defines the assistant's role and the format we want back. No interpolation
# needed here, so it's a plain string (not an f-string).
system_prompt = (
    "You are a helpful assistant that summarizes website content. "
    "Provide a concise summary of the main points and topics covered on the "
    "website. Respond in markdown format."
)

In [27]:
# The user prompt carries the scraped page text for the model to summarize.
user_prompt = f"Here is the content of the website: {my_website_content}"

In [28]:
from typing import Dict, List


def summarize_website_content(messages: List[Dict[str, str]]) -> str | None:
    """Send a prepared messages list to the model and return the reply text.

    Args:
        messages: Chat messages (system + user) describing the task and data.

    Returns:
        The model's response content, or ``None`` if the model returned no text.
    """
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages,  # type: ignore
    )
    return response.choices[0].message.content

In [29]:
# Combine both prompts into the messages list the API expects.
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [30]:
# Send the system + user prompts together and render the markdown summary.
response = summarize_website_content(messages)

display(Markdown(data=f"**GPT:** {response}"))

**GPT:** - Overview
  - Azam Mustufa, Software Engineer (AZAM.DEV)
  - Location: Bangalore, India; Open to remote
  - Currently at IBM, focusing on SAP SuccessFactors cloud integrations; backend with Java, Spring Boot, and Node.js
  - Strong emphasis on quality engineering, test automation, and distributed systems

- Key personal metrics
  - 281+ LeetCode problems solved, Top 13%
  - 5 projects deployed
  - 3× Hackathon finalist

- Experience
  - IBM · Bangalore, India (Dec 2025 — Present)
    - Software Engineer building backend services for SAP SuccessFactors cloud integrations
    - Tech: Java, Spring Data JPA, Hibernate, Spring Security + JWT
    - Async processing with @Async and @RabbitListener; transactional data handling; unit/integration tests with JUnit/Mockito
    - Achievements: resolved 30+ defects during functional testing; improved release stability by ~15%
  - Education: B.E. in Computer Science & Engineering, Visvesvaraya Technological University (Sept 2021 — June 2025)
    - CGPA: 7.28/10; core courses in OS, Networks, OOP, DBMS, DS&A, Software Engineering
    - 3× Hackathon Finalist (regional and university levels)

- Projects (highlights)
  - Database Backup CLI Infrastructure
    - Production-grade automation CLI for async database backups
    - Outcomes: reduces deployment/migration time by 80%; handles 50GB migrations under 150MB memory; 99.9% success
    - Tech: TypeScript, MongoDB, MySQL, Node.js, Linux, Bash
  - Project Management Engine
    - Scalable REST backend with 25+ endpoints; granular RBAC
    - Performance: API responses under 120ms; Redis caching; BullMQ pipelines
    - Tech: Express, MongoDB, Redis, Docker, BullMQ, JWT
  - Distributed Image Processing Backend
    - Dynamic image transformations with scalable delivery
    - Outcomes: 40% payload reduction; 25% faster queries via Mongoose schemas; streaming uploads under load
    - Tech: Bun, Node.js, Express, MongoDB, Mongoose, Sharp
  - HTTP Caching Proxy
    - CLI-driven proxy caching GET responses to disk; HIT/MISS/BYPASS headers
    - Tech: TypeScript, Node.js, Express
  - UPI Offline Mesh
    - Spring Boot backend simulating UPI payments via Bluetooth mesh (no internet)
    - Security: Hybrid RSA-OAEP + AES-GCM; atomic idempotency with ConcurrentHashMap as Redis SETNX
    - Tech: Java, Spring Boot, RSA/AES-GCM, H2, JUnit

- Technical Skills
  - Languages: Java, JavaScript, TypeScript, Python, C++, SQL
  - Frameworks/Libraries: Spring Boot, Spring Security, Node.js, Express, Bun, BullMQ, Mongoose
  - Databases/DevOps/Testing: MongoDB, MySQL, Redis, RabbitMQ, Docker, Linux, CI/CD, Git, JUnit, Mockito, Selenium, Playwright, Cypress

- Contact & Availability
  - Resume: View / Download PDF
  - Email: azammustafa66@gmail.com
  - GitHub, LinkedIn, LeetCode profiles
  - Status: Accepting new requests
  - Location: Bangalore, India; Open to Remote

- Brand
  - AZAM.DEV site built with Nuxt 3
  - Quick access to Home, Work, Projects, Experience, Contact

If you’d like, I can tailor this summary for a resume, LinkedIn About section, or a short portfolio blurb.

## 6. Reuse the helper on another page

Now that `summarize_website_content` is a function, summarizing a different
site is just: scrape → build messages → call the helper.

In [31]:
content = fetch_website_content(url="https://cnn.com/")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Here is the content of the website: {content}"},
]

In [32]:
summary = summarize_website_content(messages)
display(Markdown(data=f"**GPT:** {summary}"))

**GPT:** - Overview
  - CNN’s homepage focuses on breaking news, latest updates, and video content across global and US news, with real-time live updates, exclusive reporting, and multimedia storytelling.
  - The site offers regional editions, personalized features (topics, newsletters, account), and a broad ecosystem of shows, investigations, podcasts, and interactive tools.

- Main sections and topics (navigation highlights)
  - Categories: US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, Weather
  - Special topics: World Cup 2026; Ukraine-Russia War; Israel-Hamas War
  - Additional hubs: Games; Watch; Live TV; Listen; Features (As Equals, Call to Earth, Freedom Project, Impact Your World, Inside Africa, CNN Heroes)

- Content types you’ll find
  - News formats: Breaking News, Live Updates, CNN Exclusives, Analysis, Investigations, Profiles
  - Multimedia: Videos (including CNN Shorts), Live TV, Podcasts
  - Interactive and serialized content: Photo essays, long-form features, and special reports

- Tools and resources
  - Newsletters, topics you follow, and account features (sign in, follow, customize)
  - Interactive data and trackers (e.g., redistricting, polls, market data)
  - Games and edukational content (Crosswords, Sudoku, CNN 10)

- Accessibility and language options
  - Editions for US and International audiences, with Arabic and Español language options

- Additional elements
  - App and device access: download the CNN app and access content on the go
  - Ad feedback prompt and user interaction options displayed on the site

In short, the site is a comprehensive news platform offering real-time breaking coverage, global and US news across many categories, multimedia reporting, investigations and features, interactive tools, podcasts, games, and personalized user options.

## 7. Selenium-based web scraping

The `requests`-based scraper only sees the initial HTML. For sites that
render their content with JavaScript (single-page apps, infinite scroll,
etc.), we need a real browser. **Selenium** drives a headless Chrome instance
so we capture the fully-rendered page.

The `WebsiteSummarizer` class below bundles the whole flow — fetch, summarize,
display — behind a small object.

> **Requires:** `selenium` (already in `requirements.txt`). Selenium 4.6+ auto-downloads
> a matching chromedriver, so no manual driver setup is needed — just a local Chrome install.

In [42]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


class WebsiteSummarizer:
    """Fetch a JS-rendered page with Selenium and summarize it with an LLM.

    Typical usage:
        ws = WebsiteSummarizer(url)
        ws.selenium_fetch_content()
        ws.summarize_content()
        ws.display_summary()
    """

    def __init__(self, url: str):
        self.url = url
        self.content: str | None = None 
        self.summary: str | None = None 
        self.openai = OpenAI()

    def selenium_fetch_content(self) -> None:
        """Load the page in headless Chrome and capture the cleaned text."""
        options = Options()
        options.add_argument(
            "User-Agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        )
        driver = webdriver.Chrome(options=options)
        try:
            driver.get(url=self.url)
            driver.implicitly_wait(5)
            # Strip the rendered HTML down to title + visible text so we send far fewer tokens to the model.
            self.content = clean_html(html=driver.page_source)
        except Exception as e:
            print(f"Error fetching content with Selenium: {e}")
        finally:
            driver.quit()

    def summarize_content(self) -> None:
        """Summarize the fetched content using the LLM."""
        if not self.content:
            raise ValueError(
                "Content not fetched yet. Call selenium_fetch_content() first."
            )

        system_prompt = (
            "You are a helpful assistant that summarizes website content. "
            "Provide a concise summary of the main points and topics covered on the "
            "website. Respond in markdown format."
        )
        user_prompt = f"Here is the content of the website: {self.content}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        response = self.openai.chat.completions.create(
            model="gpt-5-nano",
            messages=messages,  # type: ignore
        )
        self.summary = response.choices[0].message.content

    def display_summary(self) -> None:
        """Render the summary as markdown."""
        if not self.summary:
            raise ValueError(
                "Summary not generated yet. Call summarize_content() first."
            )
        display(Markdown(data=self.summary))

In [45]:
ws = WebsiteSummarizer(url="https://www.netflix.com/")
ws.selenium_fetch_content()
ws.summarize_content()
ws.display_summary()

## Summary of main points and topics (Netflix India homepage)

- **What Netflix is**: A streaming service offering a wide library of TV shows, movies, documentaries, anime, and Netflix originals. Ad-free viewing with new titles added weekly.

- **Plans and pricing**: Flexible pricing from ₹149 to ₹649 per month with no cancellation fees; you can start, stop, or restart anytime.

- **How it works / where to watch**:
  - Watch on a wide range of devices (smart TVs, PlayStation, Xbox, Chromecast, Apple TV, Blu-ray players, smartphones, tablets, laptops, etc.).
  - Download shows to watch offline.
  - Watch anywhere, anytime; sign in on web or on apps.

- **Kids features**: Profiles for kids with PIN-protected parental controls to restrict content by maturity level.

- **Trending Now**: Homepage highlights popular titles (examples include Money Heist, Stranger Things, Sex/Life, etc.) and other regional picks.

- **What’s on the service**: Extensive library including feature films, documentaries, shows, anime, and Netflix originals.

- **Getting started**: Prompt to enter your email to create or restart a membership.

- **Support and policy information**:
  - FAQ and Help Centre
  - Account, Terms of Use, Privacy, Cookie Preferences
  - Contact options, including a support phone number

- **Cookie and privacy details**:
  - Page uses Google reCAPTCHA for bot protection.
  - Cookie Preferences cover Essential, First-Party Performance/Functionality, Third-Party (Tudum), and Advertising cookies.
  - Opt-out options exist but may affect service functionality; detailed explanations are provided in the Privacy Statement.

- **Language and accessibility**: Language options (English and हिंदी) available; standard accessibility and privacy notices.

In short, the page promotes Netflix India as an ad-free streaming service with flexible pricing, broad device support, offline downloads, kid-friendly profiles, and extensive help and policy information, along with a dynamic list of trending content.